# Karpathy-split evaluation — same protocol, same toolkit, same metrics as the papers

This notebook evaluates your trained checkpoint the **exact way** the classic
image-captioning papers do it, so the numbers it produces can be placed
directly next to published results:

- **Split**: the [Karpathy split](https://cs.stanford.edu/people/karpathy/deepimagesent/) of MS-COCO
  (113,287 train / 5,000 val / **5,000 test**) — the split used by *Show and
  Tell* (Vinyals et al., 2015), *Show, Attend and Tell* (Xu et al., 2015),
  and essentially every captioning paper since.
- **Decoding**: beam search (default beam width 5), not greedy — matching
  how these papers, and your own `predict.py`, actually generate captions.
- **Scoring**: the *official* [`pycocoevalcap`](https://github.com/salaniz/pycocoevalcap)
  toolkit — the Python 3 port of Microsoft's COCO Caption Evaluation code —
  using its own `PTBTokenizer`, `Bleu`, `Meteor`, `Rouge`, and `Cider`
  classes. This is the same scoring code, called the same way, that produces
  the numbers in these papers' tables.

**This notebook is entirely separate from `app.py` / `predict.py` / `train.py`.**
It only *reads* your trained checkpoint — nothing in `captioning/` is
imported for writing, and nothing about the live app or CLI changes because
of this file existing.

## Requirements (one-time)

```bash
pip install pycocoevalcap
```

You also need a **Java runtime** on your PATH — `pycocoevalcap`'s tokenizer
and METEOR scorer are wrappers around bundled Stanford NLP `.jar` files and
shell out to `java` to run them. Check with:

```bash
java -version
```

If you don't have Java, install a JRE (e.g. Eclipse Temurin, OpenJDK) —
this is a one-time system dependency, not a Python package.

## What this notebook downloads

The Karpathy split file (`dataset_coco.json`, from Andrej Karpathy's own
site) the first time you run it, cached locally afterward. It does **not**
re-download COCO images — see the note in the "Resolve local image paths"
section for why.


In [1]:

# --- Configuration -----------------------------------------------------
# Set LIMIT to a small number (e.g. 50) for a fast smoke test before
# committing to the full 5,000-image run, which can take a while on CPU.
LIMIT = None          # None = full Karpathy test split (5,000 images)
BEAM_SIZE = 5          # matches predict.py's default and common literature setting
INCLUDE_SPICE = False  # SPICE needs an extra ~2GB Stanford CoreNLP model download
                        # and is much slower; leave off unless you specifically need it


In [2]:

# --- Imports -------------------------------------------------------------
from __future__ import annotations

import io
import json
import os
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm

# Make the project package importable regardless of where this notebook
# lives, as long as it's placed at the project root (next to app.py).
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from captioning.checkpoint import load_checkpoint
from captioning.config import get_default_config
from captioning.dataset import eval_transform
from captioning.inference import beam_search_caption
from captioning.model import build_models
from captioning.vocab import Vocabulary

sys.modules["__main__"].Vocabulary = Vocabulary  # same unpickling alias predict.py/app.py use

CFG = get_default_config()
print("Project root:", PROJECT_ROOT)
print("Device:", CFG.device)


c:\Users\yokes\anaconda3\envs\image_caption\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: d:\Projects\newone
Device: cuda


In [4]:
import torch, sys
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

python: c:\Users\yokes\anaconda3\envs\image_caption\python.exe
torch: 2.5.1+cu121
cuda_available: True
device_count: 1
gpu: NVIDIA GeForce GTX 1650 Ti


In [5]:

# --- Check Java is available (required by PTBTokenizer + METEOR) --------
try:
    out = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print((out.stderr or out.stdout).strip().splitlines()[0])
except FileNotFoundError:
    raise RuntimeError(
        "Java was not found on PATH. pycocoevalcap's tokenizer and METEOR "
        "scorer need a Java runtime to run their bundled .jar files. "
        "Install a JRE (e.g. Eclipse Temurin / OpenJDK) and re-run this cell."
    )


java version "21.0.1" 2023-10-17 LTS


## 1. Download the Karpathy split

This is Andrej Karpathy's own `dataset_coco.json` — the file that defines
which 113,287 / 5,000 / 5,000 images belong to train/val/test, plus the 5
human reference captions per image. It's cached locally so this only
downloads once.


In [6]:

KARPATHY_DIR = PROJECT_ROOT / "data" / "coco" / "karpathy"
KARPATHY_JSON = KARPATHY_DIR / "dataset_coco.json"
KARPATHY_ZIP_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"

KARPATHY_DIR.mkdir(parents=True, exist_ok=True)

if not KARPATHY_JSON.is_file():
    print(f"Downloading Karpathy split from {KARPATHY_ZIP_URL} ...")
    req = urllib.request.Request(KARPATHY_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        zip_bytes = resp.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        with zf.open("dataset_coco.json") as src, open(KARPATHY_JSON, "wb") as dst:
            dst.write(src.read())
    print("Saved to", KARPATHY_JSON)
else:
    print("Already downloaded:", KARPATHY_JSON)

with open(KARPATHY_JSON, "r", encoding="utf-8") as f:
    karpathy = json.load(f)

test_entries = [im for im in karpathy["images"] if im["split"] == "test"]
print(f"Karpathy test split: {len(test_entries)} images (expect 5,000)")


Saved to d:\Projects\newone\data\coco\karpathy\dataset_coco.json
Karpathy test split: 5000 images (expect 5,000)


## 2. Resolve local image paths — reusing your existing COCO2017 folders

The Karpathy split was defined over COCO**2014**. COCO**2017** didn't add
new images — it just redrew the train/val boundary over the same underlying
image pool. So instead of re-downloading COCO2014 (several more GB), we
match each Karpathy test image by its **COCO `image_id`** against the
`image_id → file_name` map already sitting in your existing
`captions_train2017.json` / `captions_val2017.json` — files you already
have from training. If an id is found in either, we use the file that's
already on your disk in `train2017/` or `val2017/`.

If some images aren't found (a small number can be renumbered/removed
between COCO releases), they're reported and skipped rather than silently
dropped — you'll see exactly how many, and can decide whether to fill the
gap by downloading the specific missing image(s) from the COCO website.


In [7]:

def _load_id_to_file(ann_path: Path) -> dict[int, str]:
    with open(ann_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {im["id"]: im["file_name"] for im in data["images"]}

train2017_ids = _load_id_to_file(Path(CFG.train_ann))
val2017_ids = _load_id_to_file(Path(CFG.val_ann))

resolved, missing = [], []
for entry in test_entries:
    cocoid = entry["cocoid"]
    if cocoid in train2017_ids:
        local_path = Path(CFG.train_images) / train2017_ids[cocoid]
    elif cocoid in val2017_ids:
        local_path = Path(CFG.val_images) / val2017_ids[cocoid]
    else:
        missing.append(cocoid)
        continue

    if not local_path.is_file():
        missing.append(cocoid)
        continue

    refs = [s["raw"].strip() for s in entry["sentences"]]
    resolved.append({"image_id": cocoid, "path": local_path, "refs": refs})

print(f"Resolved locally: {len(resolved)} / {len(test_entries)}")
print(f"Missing (not found in your train2017/val2017): {len(missing)}")
if missing:
    print("First few missing ids:", missing[:10])

if LIMIT:
    resolved = resolved[:LIMIT]
    print(f"LIMIT set — evaluating on {len(resolved)} images only (smoke test).")


Resolved locally: 593 / 5000
Missing (not found in your train2017/val2017): 4407
First few missing ids: [391895, 60623, 483108, 384213, 386164, 223648, 294832, 462565, 436141, 192440]


### 2b. Download the missing balance (only the Karpathy-test images you lack)

You only need the ~4.4k missing **test** images, not the full ~18 GB `train2017` zip.
This cell fetches each missing file from the official COCO CDN into `train2017/` or `val2017/`, then rebuilds `resolved`.
Re-run is safe — files already on disk are skipped.

In [8]:
# --- Download only missing Karpathy-test images from COCO CDN ------------
# Official hosts: images.cocodataset.org/{train2017|val2017}/{file_name}
COCO_IMG_BASE = "http://images.cocodataset.org"
DOWNLOAD_MISSING = True   # set False to skip downloads

train_dir = Path(CFG.train_images)
val_dir = Path(CFG.val_images)
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

# Map each missing id → (local_path, url)
to_fetch = []
for cocoid in missing:
    if cocoid in train2017_ids:
        fname = train2017_ids[cocoid]
        local_path = train_dir / fname
        url = f"{COCO_IMG_BASE}/train2017/{fname}"
    elif cocoid in val2017_ids:
        fname = val2017_ids[cocoid]
        local_path = val_dir / fname
        url = f"{COCO_IMG_BASE}/val2017/{fname}"
    else:
        print(f"Skip {cocoid}: not in train2017/val2017 annotations")
        continue
    if local_path.is_file():
        continue
    to_fetch.append((cocoid, local_path, url))

print(f"Need to download: {len(to_fetch)} images")

if DOWNLOAD_MISSING and to_fetch:
    failed = []
    for cocoid, local_path, url in tqdm(to_fetch, desc="download missing"):
        try:
            urllib.request.urlretrieve(url, local_path)
        except Exception as e:
            failed.append((cocoid, str(e)))
            if local_path.exists():
                local_path.unlink()
    print(f"Downloaded OK: {len(to_fetch) - len(failed)}  ·  failed: {len(failed)}")
    if failed:
        print("First failures:", failed[:5])
elif not DOWNLOAD_MISSING:
    print("DOWNLOAD_MISSING=False — skipped.")

# Rebuild resolved / missing after downloads
resolved, missing = [], []
for entry in test_entries:
    cocoid = entry["cocoid"]
    if cocoid in train2017_ids:
        local_path = train_dir / train2017_ids[cocoid]
    elif cocoid in val2017_ids:
        local_path = val_dir / val2017_ids[cocoid]
    else:
        missing.append(cocoid)
        continue
    if not local_path.is_file():
        missing.append(cocoid)
        continue
    refs = [s["raw"].strip() for s in entry["sentences"]]
    resolved.append({"image_id": cocoid, "path": local_path, "refs": refs})

print(f"Resolved locally: {len(resolved)} / {len(test_entries)}")
print(f"Still missing: {len(missing)}")
if LIMIT:
    resolved = resolved[:LIMIT]
    print(f"LIMIT set — evaluating on {len(resolved)} images only (smoke test).")

Need to download: 4407 images


download missing: 100%|██████████| 4407/4407 [1:56:25<00:00,  1.59s/it]  


Downloaded OK: 4407  ·  failed: 0
Resolved locally: 5000 / 5000
Still missing: 0


## 3. Load the trained model

Same checkpoint auto-detection as `app.py` and `predict.py` — looks for
`checkpoints/best_model.pth`, falling back to `checkpoints/last_model.pth`.


In [9]:

def pick_checkpoint() -> Path:
    for name in ("best_model.pth", "last_model.pth"):
        p = Path(CFG.checkpoint_dir) / name
        if p.is_file():
            return p
    raise FileNotFoundError(
        f"No checkpoint found in {CFG.checkpoint_dir}/. Train the model first, "
        "or place best_model.pth / last_model.pth there."
    )

ckpt_path = pick_checkpoint()
device = torch.device(CFG.device)
state = torch.load(ckpt_path, map_location=device, weights_only=False)

vocab = state["vocab"]
saved_cfg = state.get("config") or {}
for k in ("attention_dim", "embed_dim", "decoder_dim", "encoder_dim", "dropout", "max_caption_len"):
    if k in saved_cfg:
        setattr(CFG, k, saved_cfg[k])

encoder, decoder = build_models(CFG, len(vocab), device, pretrained=False)
load_checkpoint(str(ckpt_path), encoder, decoder, map_location=device)
encoder.eval()
decoder.eval()

print(f"Loaded {ckpt_path.name}  ·  device={device.type}  ·  vocab={len(vocab):,} words")


Loaded best_model.pth  ·  device=cuda  ·  vocab=10,307 words


## 4. Generate captions with beam search over the test split

This is the slow part — one forward pass + beam search per image. Progress
is shown live; you can interrupt and re-run with a smaller `LIMIT` above to
gauge timing before committing to the full 5,000-image run.


In [10]:

gts_raw: dict[str, list[dict]] = {}
res_raw: dict[str, list[dict]] = {}

t0 = time.time()
with torch.no_grad():
    for item in tqdm(resolved, desc=f"beam_k{BEAM_SIZE}"):
        img_id = str(item["image_id"])
        image = Image.open(item["path"]).convert("RGB")
        tensor = eval_transform(image)

        caption, _ids, _alphas = beam_search_caption(
            encoder, decoder, vocab, tensor, device, CFG.encoder_dim,
            beam_size=BEAM_SIZE, max_len=CFG.max_caption_len,
        )

        res_raw[img_id] = [{"caption": caption}]
        gts_raw[img_id] = [{"caption": r} for r in item["refs"]]

elapsed = time.time() - t0
print(f"Generated {len(res_raw)} captions in {elapsed/60:.1f} min "
      f"({elapsed/max(len(res_raw),1):.2f}s/image)")


beam_k5: 100%|██████████| 5000/5000 [11:23<00:00,  7.32it/s]

Generated 5000 captions in 11.4 min (0.14s/image)


## 5. Score with the official toolkit

Same classes, same call pattern as `pycocoevalcap`'s own `COCOEvalCap.evaluate()`
(the code every paper's evaluation ultimately runs): tokenize with the
Stanford `PTBTokenizer`, then `Bleu`, `Meteor`, `Rouge`, `Cider` (and
optionally `Spice`).


In [11]:

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

print("Tokenizing with Stanford PTBTokenizer...")
tokenizer = PTBTokenizer()
gts = tokenizer.tokenize(gts_raw)
res = tokenizer.tokenize(res_raw)

results = {}

print("Scoring BLEU-1..4 ...")
bleu_scores, _ = Bleu(4).compute_score(gts, res, verbose=0)
for i, s in enumerate(bleu_scores, start=1):
    results[f"BLEU-{i}"] = s

print("Scoring METEOR ...")
results["METEOR"], _ = Meteor().compute_score(gts, res)

print("Scoring ROUGE-L ...")
results["ROUGE-L"], _ = Rouge().compute_score(gts, res)

print("Scoring CIDEr ...")
results["CIDEr"], _ = Cider().compute_score(gts, res)

if INCLUDE_SPICE:
    from pycocoevalcap.spice.spice import Spice
    print("Scoring SPICE (slower, downloads Stanford CoreNLP models on first use) ...")
    results["SPICE"], _ = Spice().compute_score(gts, res)

print()
print("=" * 40)
for k, v in results.items():
    print(f"{k:10s}: {v:.4f}")
print("=" * 40)


Tokenizing with Stanford PTBTokenizer...
Scoring BLEU-1..4 ...
Scoring METEOR ...
Scoring ROUGE-L ...
Scoring CIDEr ...

BLEU-1    : 0.7946
BLEU-2    : 0.6511
BLEU-3    : 0.5226
BLEU-4    : 0.4165
METEOR    : 0.2921
ROUGE-L   : 0.5984
CIDEr     : 1.2757


## 6. Compare against the published papers

Reference rows below are the commonly-cited Karpathy-test-split numbers for
*Show and Tell* (Google NIC) and *Show, Attend and Tell* (hard attention),
as reported in later papers' comparison tables (the standard citation
format used across the field once evaluation protocols converged).


In [12]:

import pandas as pd

published = {
    "Show and Tell (NIC, 2015)":            {"BLEU-4": 0.183, "METEOR": 0.237, "CIDEr": 0.855},
    "Show, Attend and Tell — hard (2015)":  {"BLEU-4": 0.250, "METEOR": 0.230, "CIDEr": None},
}

rows = []
for name, m in published.items():
    rows.append({"model": name, "BLEU-4": m["BLEU-4"], "METEOR": m["METEOR"], "CIDEr": m["CIDEr"]})
rows.append({
    "model": f"This model (checkpoint: {ckpt_path.name}, beam={BEAM_SIZE})",
    "BLEU-4": round(results["BLEU-4"], 3),
    "METEOR": round(results["METEOR"], 3),
    "CIDEr": round(results["CIDEr"], 3),
})

df = pd.DataFrame(rows).set_index("model")
df


,BLEU-4,METEOR,CIDEr
model,,,
"Show and Tell (NIC, 2015)",0.183,0.237,0.855
"Show, Attend and Tell — hard (2015)",0.250,0.230,NaN
"This model (checkpoint: best_model.pth, beam=5)",0.416,0.292,1.276


## 7. Save results

Written to `outputs/eval_karpathy/` — separate from `outputs/predictions/`
(the live app/CLI's output folder) so this one-off research artifact never
mixes with UI or CLI-generated files.


In [13]:

out_dir = PROJECT_ROOT / "outputs" / "eval_karpathy"
out_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "checkpoint": ckpt_path.name,
    "beam_size": BEAM_SIZE,
    "num_images_evaluated": len(resolved),
    "num_images_missing_locally": len(missing),
    "scores": results,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

out_path = out_dir / f"karpathy_eval_{time.strftime('%Y%m%d_%H%M%S')}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", out_path)


Saved: d:\Projects\newone\outputs\eval_karpathy\karpathy_eval_20260804_220304.json
